In [ ]:
import pandas as pd
import time
import os
import requests
import random
from datetime import datetime, timedelta

# ---------- 配置 ----------
API_KEYS = ["F336TKOPNHPECGEJ", "208RZJ4S5KG1B2Q9"]
key_index = 0
cache_dir = "alpha_cache"
os.makedirs(cache_dir, exist_ok=True)

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
start_date = pd.to_datetime("2013-01-01")
end_date = pd.to_datetime("2025-12-31")

indicators = ["SMA","EMA","RSI","MACD","BBANDS","ATR","STOCH"]

# ---------- Key 使用计数 ----------
key_usage = {key: 0 for key in API_KEYS}
DAILY_LIMIT = 25  # 每个 Key 每天最大请求

def check_key():
    global key_index
    for _ in range(len(API_KEYS)):
        key = API_KEYS[key_index]
        if key_usage[key] < DAILY_LIMIT:
            key_usage[key] += 1
            return key
        key_index = (key_index + 1) % len(API_KEYS)
    # 所有 Key 当天都用完，等待到明天
    now = datetime.now()
    tomorrow = datetime(now.year, now.month, now.day) + timedelta(days=1)
    wait_sec = (tomorrow - now).total_seconds() + 5
    print(f"⚠️ 所有 Key 当天请求已用完，等待 {int(wait_sec)} 秒")
    time.sleep(wait_sec)
    # 重置计数并返回当前 Key
    for k in key_usage:
        key_usage[k] = 0
    return check_key()

# ---------- 通用下载函数 ----------
def safe_download(url, params, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=20)
            data = r.json()
            if "Note" in data:  # API 超频提示
                wait = 60 + random.randint(1,5)
                print(f"⚠️ API 限制: 等待 {wait} 秒再试")
                time.sleep(wait)
                continue
            if "Error Message" in data:
                print(f"❌ 请求错误: {data['Error Message']}")
                return None
            return data
        except Exception as e:
            wait = 2 ** attempt
            print(f"❌ 请求失败: {e}, 等待 {wait} 秒重试")
            time.sleep(wait)
    return None

# ---------- 下载日线 ----------
def download_alpha_daily(ticker):
    cache_file = os.path.join(cache_dir, f"{ticker}_daily.csv")
    if os.path.exists(cache_file):
        df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
        return df[(df.index >= start_date) & (df.index <= end_date)]

    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY_ADJUSTED",
        "symbol": ticker,
        "outputsize": "full",
        "apikey": check_key()
    }
    data = safe_download(url, params)
    if not data or "Time Series (Daily)" not in data:
        print(f"❌ {ticker} 日线下载失败: {list(data.keys()) if data else None}")
        return None

    df = pd.DataFrame.from_dict(data["Time Series (Daily)"], orient="index")
    df = df.rename(columns={
        "1. open": "Open",
        "2. high": "High",
        "3. low": "Low",
        "4. close": "Close",
        "6. volume": "Volume"
    })[['Open','High','Low','Close','Volume']].astype(float)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df[(df.index >= start_date) & (df.index <= end_date)]
    df.to_csv(cache_file)
    return df

# ---------- 下载指标 ----------
def download_indicator(ticker, indicator, time_period=14):
    cache_file = os.path.join(cache_dir, f"{ticker}_{indicator}_{time_period}.csv")
    if os.path.exists(cache_file):
        df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
        return df[(df.index >= start_date) & (df.index <= end_date)]

    url = "https://www.alphavantage.co/query"
    params = {
        "function": indicator,
        "symbol": ticker,
        "interval": "daily",
        "series_type": "close",
        "apikey": check_key()
    }
    if indicator not in ["SMA","EMA","RSI","ATR"]:
        params.pop("series_type", None)
        params.pop("time_period", None)
    else:
        params["time_period"] = time_period

    data = safe_download(url, params)
    if not data:
        print(f"❌ {ticker} {indicator} 下载失败")
        return None

    key_name = [k for k in data.keys() if k not in ["Meta Data","Information","Note","Error Message"]]
    if not key_name:
        print(f"⚠️ {ticker} {indicator} 数据异常")
        return None

    df = pd.DataFrame.from_dict(data[key_name[0]], orient="index").astype(float)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df[(df.index >= start_date) & (df.index <= end_date)]
    df.to_csv(cache_file)
    return df

# ---------- 下载和合并 ----------
all_data = []
valid_tickers = []

for ticker in tickers:
    print(f"⬇️ 下载 {ticker} 日线数据")
    df_daily = download_alpha_daily(ticker)
    if df_daily is None:
        continue

    print(f"⬇️ 下载 {ticker} 技术指标")
    dfs = []
    for ind in indicators:
        df_ind = download_indicator(ticker, ind)
        if df_ind is not None:
            if ind == "SMA": df_ind.columns = [f"{ticker}_SMA"]
            elif ind == "EMA": df_ind.columns = [f"{ticker}_EMA"]
            elif ind == "RSI": df_ind.columns = [f"{ticker}_RSI"]
            elif ind == "MACD": df_ind.columns = [f"{ticker}_MACD", f"{ticker}_MACD_signal", f"{ticker}_MACD_hist"]
            elif ind == "BBANDS": df_ind.columns = [f"{ticker}_BBU", f"{ticker}_BBM", f"{ticker}_BBL"]
            elif ind == "ATR": df_ind.columns = [f"{ticker}_ATR"]
            elif ind == "STOCH": df_ind.columns = [f"{ticker}_STOCHK", f"{ticker}_STOCHD"]
            dfs.append(df_ind)

    df_daily.columns = [f"{ticker}_{c}" for c in df_daily.columns]
    df_merged = pd.concat([df_daily] + dfs, axis=1)
    df_merged = df_merged.sort_index()
    all_data.append(df_merged)
    valid_tickers.append(ticker)

    sleep_time = random.randint(12, 15)
    print(f"⏳ 等待 {sleep_time} 秒再下载下一支股票")
    time.sleep(sleep_time)

# ---------- 多股票合并 ----------
data = pd.concat(all_data, axis=1).dropna()
print("🎉 数据合并完成，shape:", data.shape)

# ---------- 生成交叉特征 ----------
for i in range(len(valid_tickers)):
    for j in range(i+1, len(valid_tickers)):
        t1, t2 = valid_tickers[i], valid_tickers[j]
        data[f'{t1}_{t2}_ratio'] = data[f'{t1}_Close'] / data[f'{t2}_Close']
        data[f'{t1}_{t2}_diff'] = data[f'{t1}_Close'] - data[f'{t2}_Close']
        data[f'{t1}_{t2}_corr20'] = data[f'{t1}_Close'].rolling(20).corr(data[f'{t2}_Close'])

# ---------- 保存 CSV ----------
output_file = "multi_stock_full_features_2013_2025_safe.csv"
data.to_csv(output_file)
print(f"✅ 最终数据已保存：{output_file}")